In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ===== Paths (ikut project anda) =====
DATA_PATH = r"C:\xampp\htdocs\Smart Property Valuer FYP\dataset_with_url.csv"
MODEL_PATH = r"C:\xampp\htdocs\Smart Property Valuer FYP\models\best_model.pkl"
COL_PATH = r"C:\xampp\htdocs\Smart Property Valuer FYP\models\model_columns.pkl"

# ===== Load =====
df = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)
feature_columns = joblib.load(COL_PATH)

# ===== Prepare like app.py =====
work = df.copy()
work["Price"] = pd.to_numeric(work["Price"], errors="coerce")
work["negeri"] = work["Negeri"].astype(str).str.strip() if "Negeri" in work.columns else work["negeri"].astype(str).str.strip()
work = work.dropna(subset=["Price"])

# numeric columns used by app
MODEL_NUMERIC_COLUMNS = [
    "Built_Up_SF", "Bathroom", "Furnishing", "Bedroom",
    "Tenure", "Car_Park", "Property_Type", "Land_Size"
]

x_df = pd.DataFrame(0.0, index=work.index, columns=feature_columns, dtype=float)

for col in MODEL_NUMERIC_COLUMNS:
    if col in x_df.columns and col in work.columns:
        x_df[col] = pd.to_numeric(work[col], errors="coerce").fillna(0.0)

for col in feature_columns:
    if str(col).startswith("negeri_"):
        state_name = str(col).replace("negeri_", "", 1)
        x_df[col] = (work["negeri"] == state_name).astype(float)

y_true = work["Price"].astype(float).values
y_pred = model.predict(x_df)

# ===== Metrics =====
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

safe_y = np.where(y_true <= 0, 1.0, y_true)
mape = np.mean(np.abs((y_true - y_pred) / safe_y)) * 100.0
estimated_accuracy = max(0.0, min(100.0, 100.0 - mape))

print(f"Sample size           : {len(y_true)}")
print(f"MAE                   : {mae:,.2f}")
print(f"RMSE                  : {rmse:,.2f}")
print(f"R²                    : {r2:.4f}")
print(f"MAPE (%)              : {mape:.2f}")
print(f"Estimated Accuracy (%) : {estimated_accuracy:.2f}")
print("\nCheck:")
print(f"100 - MAPE = {100 - mape:.2f}")

Sample size           : 5981
MAE                   : 70,414.45
RMSE                  : 98,729.84
R²                    : 0.8105
MAPE (%)              : 17.10
Estimated Accuracy (%) : 82.90

Check:
100 - MAPE = 82.90
